# Business Entity Resolution — Colab runner

Runtime: **High-RAM**, GPU (A100/L4) for later dense steps.

One-time setup on Google Drive: put the challenge zip at `MyDrive/amazon_er/student_resource.zip`.
Work files are cached in `MyDrive/amazon_er/work`, so every step skips itself if its output already exists.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/amazon_er'
ZIP = f'{DRIVE}/student_resource.zip'
REPO = '/content/SOMETHINGNEW_MLCODE'
DATA = '/content/student_resource/dataset'
WORK = f'{DRIVE}/work'
os.makedirs(WORK, exist_ok=True)
assert os.path.exists(ZIP), f'Upload the challenge zip to {ZIP}'

In [ ]:
# Clone or update the code. For a private repo, add a GitHub token as the Colab secret GITHUB_TOKEN.
import subprocess
url = 'https://github.com/kunal12kumar/SOMETHINGNEW_MLCODE.git'
try:
    from google.colab import userdata
    url = url.replace('https://', f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', url, REPO], check=True)
%cd {REPO}
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt
if not os.path.exists(DATA):
    !unzip -q -o "{ZIP}" -d /content && rm -rf /content/__MACOSX
!ls {DATA}/train {DATA}/test
!nproc && free -g

In [ ]:
import os
os.environ['PYTHONPATH'] = f'{REPO}/src'
!python -m pytest -q tests

In [ ]:
WORKERS = os.cpu_count()
!python -m ber.preprocess --data-dir {DATA} --out-dir {WORK}/clean --workers {WORKERS}
!python -m ber.fill_state --clean-dir {WORK}/clean
!python -m ber.splits --clean-dir {WORK}/clean --out {WORK}/splits.parquet
!python -m ber.state_neighbors --data-dir {DATA} --clean-dir {WORK}/clean --out {WORK}/state_neighbors.json

## 2. Candidates for training (150k train + 30k validation S1)

In [ ]:
!python -m ber.run_candidates --split train --clean-dir {WORK}/clean --splits {WORK}/splits.parquet \
    --n-train 150000 --n-valid 30000 --threads {WORKERS} --neighbors {WORK}/state_neighbors.json --out {WORK}/cands_train.parquet

## 3. Train the matcher and tune the threshold on validation macro F0.5

In [ ]:
!python -m ber.matcher train --data-dir {DATA} --clean-dir {WORK}/clean --cands {WORK}/cands_train.parquet --model-dir {WORK}/model

## 4. Test candidates, prediction, validation of the output files

In [ ]:
!python -m ber.run_candidates --split test --clean-dir {WORK}/clean --threads {WORKERS} \
    --neighbors {WORK}/state_neighbors.json --out {WORK}/cands_test.parquet
!python -m ber.predict --clean-dir {WORK}/clean --cands {WORK}/cands_test.parquet --model-dir {WORK}/model --out-dir {DRIVE}/output
!python /content/student_resource/utils/validate_submission.py --matching {DRIVE}/output/matching_results.tsv \
    --candidate {DRIVE}/output/candidate_pairs.tsv --test-dir {DATA}/test